In [1]:
import polars as pl


DATASET_NAME_PERFIX = "dataset_255w_"
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1


df_with_parent= pl.read_csv(
    "lehner_dataset_with_sequences_normalized.csv",
    separator=",",
    null_values="NA",
    infer_schema_length=10000
)
df_with_parent = df_with_parent.drop_nulls(subset=["original_sequence", "position", "normalized_fitness_sigma"])

df_with_parent = df_with_parent.drop(["domain_ID", "aa_seq", "input_count_rep1", "input_count_rep2", "input_count_rep3",
                 "output_count_rep1", "output_count_rep2", "output_count_rep3",
                 "mean_input_count", "quality_rank", "fitness", 'fitness_sigma'])

df_with_parent

uniprot_ID,wt_aa,position,mut_aa,STOP,normalized_fitness,normalized_fitness_sigma,original_sequence,reverse,normalized_fitness_sigmoid
str,str,i64,str,bool,f64,f64,str,bool,f64
"""A0A2R8Y422""","""*""",2,"""Q""",true,0.81905,0.208478,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.36025
"""A0A2R8Y422""","""A""",2,"""Q""",false,0.28079,0.093461,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.128588
"""A0A2R8Y422""","""C""",2,"""Q""",false,0.10325,0.057995,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.047511
"""A0A2R8Y422""","""D""",2,"""Q""",false,0.258003,0.072296,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,0.118255
"""A0A2R8Y422""","""E""",2,"""Q""",false,-0.376783,0.169263,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",true,-0.086631
…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0""","""C""",1059,"""V""",false,-1.113722,0.157535,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.251218
"""Q9Y6V0""","""C""",1059,"""W""",false,-0.903822,0.15048,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.205368
"""Q9Y6V0""","""C""",1059,"""Y""",false,-1.149495,0.155403,"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…",false,-0.258928


In [2]:
df_with_parent["original_sequence"].str.len_chars().describe()

statistic,value
str,f64
"""count""",1.116004e6
"""null_count""",0.0
"""mean""",911.513955
"""std""",838.009222
"""min""",66.0
"""25%""",422.0
"""50%""",683.0
"""75%""",1099.0
"""max""",8525.0


In [3]:
# adding the columns with the fragment of 255 aa around the mutation


# Remove rows with NaN in original_sequence and position
df_127 = df_with_parent.drop_nulls(subset=["original_sequence", "position", "normalized_fitness", "normalized_fitness_sigma"])


# Add sub_sequence column (21 aa from 'position')
df_127 = df_127.with_columns(
    pl.col("original_sequence").str.slice(pl.col("position").cast(pl.Int64), 21).alias("sub_sequence")
)

WINDOW = 255
HALF = WINDOW // 2  # 256

def extract_fragment(seq: str, pos: int, mut_aa=None) -> tuple[str, int]:
    if seq is None or pos is None:
        return None, None

    pos -= 1  # Convert to 0-based index
    seq_len = len(seq)
    # Navrhujeme start/end tak, aby pos byl uprostřed
    start = max(0, pos - HALF)
    end = min(seq_len, start + WINDOW)

    # Pokud na konci nezbyde 512 znaků, posuň začátek zpět
    if end - start < WINDOW:
        start = max(0, end - WINDOW)

    if mut_aa:
        seqlist = list(seq)
        seqlist[pos] = mut_aa
        seq = "".join(seqlist)

    fragment = seq[start:end]
    local_pos = pos - start
    return fragment, local_pos

def position_normilized(seq: str, pos: int) -> int:
    if seq is None or pos is None:
        return None
    pos -= 1  # Convert to 0-based index
    seq_len = len(seq)
    if pos < 0 or pos >= seq_len:
        return None
    return pos/ seq_len

df_127 = df_127.with_columns([
    pl.struct(["original_sequence", "position", "mut_aa"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]), x["mut_aa"])[0],
                    return_dtype=pl.Utf8)
      .alias("fragment_255_mut"),

    pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]))[1],
                    return_dtype=pl.Int64)
      .alias("local_position"),

    pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: extract_fragment(x["original_sequence"], int(x["position"]))[0],
                    return_dtype=pl.Utf8)
      .alias("fragment_255_org"),

        pl.struct(["original_sequence", "position"])
      .map_elements(lambda x: position_normilized(x["original_sequence"], int(x["position"])),
                    return_dtype=pl.Float32)
      .alias("position_normalized"),
])


df_127 = df_127.drop(["original_sequence", "sub_sequence", "position", "normalized_fitness", "normalized_fitness_sigma"])

df_127


uniprot_ID,wt_aa,mut_aa,STOP,reverse,normalized_fitness_sigmoid,fragment_255_mut,local_position,fragment_255_org,position_normalized
str,str,str,bool,bool,f64,str,i64,str,f32
"""A0A2R8Y422""","""*""","""Q""",true,true,0.36025,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""A""","""Q""",false,true,0.128588,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""C""","""Q""",false,true,0.047511,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""D""","""Q""",false,true,0.118255,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
"""A0A2R8Y422""","""E""","""Q""",false,true,-0.086631,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",1,"""MQIFVKTLMGKTITLEVELSDTIDNVKAKI…",0.00641
…,…,…,…,…,…,…,…,…,…
"""Q9Y6V0""","""C""","""V""",false,false,-0.251218,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757
"""Q9Y6V0""","""C""","""W""",false,false,-0.205368,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757
"""Q9Y6V0""","""C""","""Y""",false,false,-0.258928,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",127,"""FGFGASIFSQASNLISTAGQPGPHSQSGPG…",0.205757


In [4]:
# First, we shuffle the data randomly
df_shuffled = df_127.sample(fraction=1, shuffle=True, seed=123) # Using a seed for reproducibility

# Get the total number of rows
n_rows = len(df_shuffled)

# Calculate the sizes of each set (80% training, 10% validation, 10% testing)
train_size = int(TRAIN_SIZE * n_rows)
val_size = int(VAL_SIZE * n_rows)
# The remainder will be the testing set

# Split the dataframe into three parts
train_df = df_shuffled.slice(0, train_size)
val_df = df_shuffled.slice(train_size, val_size)
test_df = df_shuffled.slice(train_size + val_size) # to the end

test_df_noreverse = test_df.filter(pl.col("reverse") == False)


# Save each dataset to a separate CSV file
train_df.write_csv(f"{DATASET_NAME_PERFIX}train.csv")
val_df.write_csv(f"{DATASET_NAME_PERFIX}validation.csv")
test_df.write_csv(f"{DATASET_NAME_PERFIX}test.csv")
test_df_noreverse.write_csv(f"{DATASET_NAME_PERFIX}_noreverse_test.csv")

print(f"Datasets were successfully created and saved:")
print(f"- Training set (training_dataset.csv): {len(train_df)} rows")
print(f"- Validation set (validation_dataset.csv): {len(val_df)} rows")
print(f"- Testing set (testing_dataset.csv): {len(test_df)} rows")
print(f"- Testing set without reverse mutations (testing_noreverse_dataset.csv): {len(test_df_noreverse)} rows")



Datasets were successfully created and saved:
- Training set (training_dataset.csv): 892803 rows
- Validation set (validation_dataset.csv): 111600 rows
- Testing set (testing_dataset.csv): 111601 rows
- Testing set without reverse mutations (testing_noreverse_dataset.csv): 55760 rows


In [18]:
train_df

uniprot_ID,wt_aa,mut_aa,STOP,reverse,normalized_fitness_sigmoid,fragment_255_mut,local_position,fragment_255_org,position_normalized
str,str,str,bool,bool,f64,str,i64,str,f32
"""P48431""","""*""","""K""",true,true,0.564912,"""MYNMMETELKPPGPQQTSGGGGGNSTAAAA…",79,"""MYNMMETELKPPGPQQTSGGGGGNSTAAAA…",0.249211
"""Q13009""","""M""","""D""",false,true,-0.022803,"""TARDTLELICKTHQLDHSAHYLRLKFLIEN…",127,"""TARDTLELICKTHQLDHSAHYLRLKFLIEN…",0.573224
"""P46937""","""K""","""G""",false,true,0.045276,"""SHSRQASTDAGTAGALTPQHVRAHSSPASL…",127,"""SHSRQASTDAGTAGALTPQHVRAHSSPASL…",0.454365
"""Q9Y5K6""","""P""","""R""",false,true,-0.046137,"""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…",58,"""MVDYIVEYDYDAVHDDELTIRVGEIIRNVK…",0.090767
"""P10242""","""L""","""T""",false,true,0.191245,"""KLKKLVEQNGTDDWKVIANYLPNRTDVQCQ…",127,"""KLKKLVEQNGTDDWKVIANYLPNRTDVQCQ…",0.275
…,…,…,…,…,…,…,…,…,…
"""P25685""","""V""","""S""",false,true,0.032842,"""MGKDYYQTLGLARGASDEEIKRAYRRQALR…",55,"""MGKDYYQTLGLARGASDEEIKRAYRRQALR…",0.161765
"""O15405""","""R""","""G""",false,true,0.125761,"""PPAQLTTINQSQLSAQLGLNLGGASMPHTS…",127,"""PPAQLTTINQSQLSAQLGLNLGGASMPHTS…",0.513889
"""Q13501""","""K""","""D""",false,true,0.097384,"""RKVKHGHFGWPGWEMGPPGNWSPRPPRAGE…",237,"""RKVKHGHFGWPGWEMGPPGNWSPRPPRAGE…",0.959091
